<div style="display:flex; align-items:center; gap:10px; margin-bottom:8px;">
  <span style="font-size:26px; color:#9558B2;">●</span>
  <span style="font-size:26px; color:#389826;">●</span>
  <span style="font-size:26px; color:#CB3C33;">●</span>
  <span style="font-size:26px; color:#4063D8;">●</span>
  <span style="font-size:30px; font-weight:700; margin-left:6px;">Julia</span>
</div>

# Julia od zera — **Lesson 14**
## 📘 **Final Project / Capstone**
### **ThermalLab — numeryczny analizator procesu chłodzenia**

**Cartesian School · Julia Course**  
**Autor:** Siergej Sobolewski  
**Copyright:** © 2026 Cartesian School


## Informacje o lekcji

| Pole | Wartość |
|---|---|
| Kurs | Julia od zera |
| Numer lekcji | Lesson 14 |
| Typ | Final Project / Capstone |
| Poziom | Średniozaawansowany |
| Szacowany czas | 6–10 godzin |
| Wymagania | Lesson 0–13 |
| Główne obszary | funkcje, struktury danych, plotting, multiple dispatch, pakiety, wydajność, algebra liniowa, metody numeryczne |
| Rezultat | kompletny mini-system analityczno-symulacyjny |
| Autor | Siergej Sobolewski |
| Prawa | © 2026 Cartesian School |


## Cel Capstone

ThermalLab integruje cały kurs w jednym projekcie.

System:

1. reprezentuje fizyczny model chłodzenia;
2. przechowuje walidowane dane pomiarowe;
3. symuluje dynamikę układu;
4. obsługuje różne integratory przez multiple dispatch;
5. estymuje parametr modelu przez least squares;
6. liczy residuals i metryki błędu;
7. benchmarkuje implementacje;
8. opcjonalnie wizualizuje dane;
9. eksportuje wyniki;
10. posiada testy.


## Powiązanie z całym kursem

| Temat | Wykorzystanie |
|---|---|
| Strings | raporty i formatowanie |
| Data Structures | `struct`, `Vector`, `NamedTuple` |
| Loops | symulacja |
| Conditionals | walidacja |
| Functions | architektura funkcjonalna |
| Packages | `Pkg`, `LinearAlgebra`, `Statistics`, opcjonalnie `Plots` |
| Plotting | wykresy pomiarów i residuals |
| Multiple Dispatch | Euler / RK4 |
| Julia is Fast | timing i alokacje |
| Linear Algebra | least squares |
| Factorizations | operator `\` |
| Numerical Computing | ODE, błąd, walidacja |


## Architektura projektu

```text
ThermalLab
├── ThermalModel
├── MeasurementSet
├── SimulationResult
├── AbstractIntegrator
│   ├── EulerIntegrator
│   └── RK4Integrator
├── simulate(...)
├── estimate_k(...)
├── metrics(...)
├── benchmark
├── plotting
├── export
└── tests
```


# Część I — środowisko


## **1. Pakiety standardowe**


In [ ]:
using LinearAlgebra
using Statistics
using Random
using Printf
using Pkg

Random.seed!(2026)

println("Aktywny projekt: ", Pkg.project().path)


## **2. Opcjonalne `Plots.jl`**


In [ ]:
const HAS_PLOTS = try
    @eval using Plots
    true
catch
    false
end

println("Plots.jl dostępny: ", HAS_PLOTS)


### Ważne

Notebook nie wykonuje automatycznie `Pkg.add("Plots")`.

Jeśli chcesz wykresy:

```text
pkg> add Plots
```


# Część II — model fizyczny


## **3. Prawo chłodzenia Newtona**

\[
\frac{dT}{dt} = -k(T-T_{env})
\]

Rozwiązanie analityczne:

\[
T(t)=T_{env}+(T_0-T_{env})e^{-kt}
\]


## **4. `ThermalModel`**


In [ ]:
struct ThermalModel{T<:Real}
    k::T
    ambient::T

    function ThermalModel(k::T, ambient::T) where {T<:Real}
        k > zero(T) || throw(ArgumentError("k musi być dodatnie"))
        new{T}(k, ambient)
    end
end


In [ ]:
model = ThermalModel(0.12, 20.0)
@show model


## **5. `MeasurementSet`**


In [ ]:
struct MeasurementSet{T<:Real}
    time::Vector{T}
    temperature::Vector{T}

    function MeasurementSet(time::Vector{T}, temperature::Vector{T}) where {T<:Real}
        length(time) == length(temperature) ||
            throw(ArgumentError("time i temperature muszą mieć tę samą długość"))

        isempty(time) &&
            throw(ArgumentError("zbiór nie może być pusty"))

        issorted(time) ||
            throw(ArgumentError("czas musi być posortowany"))

        new{T}(time, temperature)
    end
end


## **6. `SimulationResult`**


In [ ]:
struct SimulationResult{T<:Real}
    time::Vector{T}
    temperature::Vector{T}
    method::Symbol
end


### Analiza architektury

Oddzielamy:

- model,
- dane,
- wynik symulacji.

Dzięki temu każdy typ ma jedną odpowiedzialność.


# Część III — funkcje modelu


## **7. Równanie stanu i rozwiązanie dokładne**


In [ ]:
thermal_rhs(model::ThermalModel, t, T) =
    -model.k * (T - model.ambient)

exact_temperature(model::ThermalModel, T0, t) =
    model.ambient + (T0 - model.ambient) * exp(-model.k * t)


In [ ]:
@show thermal_rhs(model, 0.0, 90.0)
@show exact_temperature(model, 90.0, 10.0)


# Część IV — multiple dispatch


## **8. Abstrakcja integratora**


In [ ]:
abstract type AbstractIntegrator end

struct EulerIntegrator <: AbstractIntegrator end
struct RK4Integrator <: AbstractIntegrator end


## **9. `step` dla Eulera**


In [ ]:
function step(::EulerIntegrator, model::ThermalModel, t::Real, T::Real, h::Real)
    return T + h * thermal_rhs(model, t, T)
end


## **10. `step` dla RK4**


In [ ]:
function step(::RK4Integrator, model::ThermalModel, t::Real, T::Real, h::Real)
    k1 = thermal_rhs(model, t, T)
    k2 = thermal_rhs(model, t + h/2, T + h*k1/2)
    k3 = thermal_rhs(model, t + h/2, T + h*k2/2)
    k4 = thermal_rhs(model, t + h, T + h*k3)

    return T + h * (k1 + 2k2 + 2k3 + k4) / 6
end


In [ ]:
@show methods(step)


### Dlaczego multiple dispatch?

Nie używamy warunku tekstowego typu `if method == "euler"`.

Silnik wybiera odpowiednią metodę `step` na podstawie typu integratora.


# Część V — silnik symulacji


## **11. `simulate`**


In [ ]:
function simulate(
    model::ThermalModel,
    integrator::AbstractIntegrator,
    T0::Real,
    tspan::Tuple{<:Real,<:Real},
    h::Real,
)
    t0, t1 = tspan

    h > 0 || throw(ArgumentError("h musi być dodatnie"))
    t1 > t0 || throw(ArgumentError("t1 musi być większe od t0"))

    n = Int(floor((t1 - t0) / h))

    times = Vector{Float64}(undef, n + 1)
    temperatures = Vector{Float64}(undef, n + 1)

    times[1] = float(t0)
    temperatures[1] = float(T0)

    for i in 1:n
        t = times[i]
        T = temperatures[i]

        times[i + 1] = t + h
        temperatures[i + 1] = step(integrator, model, t, T, h)
    end

    method = integrator isa EulerIntegrator ? :euler : :rk4

    return SimulationResult(times, temperatures, method)
end


In [ ]:
result_euler = simulate(model, EulerIntegrator(), 90.0, (0.0, 30.0), 0.1)
result_rk4   = simulate(model, RK4Integrator(),   90.0, (0.0, 30.0), 0.1)

@show result_euler.temperature[end]
@show result_rk4.temperature[end]


## **12. Rozwiązanie dokładne dla serii czasu**


In [ ]:
exact_series(model::ThermalModel, T0, times) =
    [exact_temperature(model, T0, t) for t in times]

exact_rk4 = exact_series(model, 90.0, result_rk4.time)


# Część VI — metryki


## **13. Residuals, MAE, RMSE**


In [ ]:
residuals(predicted, observed) = predicted .- observed

mae(predicted, observed) =
    mean(abs.(residuals(predicted, observed)))

rmse(predicted, observed) =
    sqrt(mean(abs2, residuals(predicted, observed)))


In [ ]:
@show mae(result_euler.temperature, exact_rk4)
@show rmse(result_euler.temperature, exact_rk4)
@show mae(result_rk4.temperature, exact_rk4)
@show rmse(result_rk4.temperature, exact_rk4)


## **14. Walidacja przewagi RK4**


In [ ]:
@assert rmse(result_rk4.temperature, exact_rk4) <
        rmse(result_euler.temperature, exact_rk4)

println("PASS — RK4 dokładniejsze od Eulera dla tego kroku.")


# Część VII — dane pomiarowe


## **15. Generowanie syntetycznych danych**


In [ ]:
function generate_measurements(
    model::ThermalModel,
    T0::Real,
    times::Vector{Float64};
    noise_std::Real=0.4,
    rng=Random.default_rng(),
)
    noise_std >= 0 ||
        throw(ArgumentError("noise_std nie może być ujemne"))

    exact = exact_series(model, T0, times)
    noisy = exact .+ noise_std .* randn(rng, length(times))

    return MeasurementSet(copy(times), noisy)
end


In [ ]:
measurement_times = collect(0.0:1.0:30.0)

data = generate_measurements(
    model,
    90.0,
    measurement_times;
    noise_std=0.35,
)

@show length(data.time)
@show data.temperature[1:5]


# Część VIII — algebra liniowa i least squares


## **16. Linearyzacja modelu**

\[
T(t)-T_{env}=(T_0-T_{env})e^{-kt}
\]

Po logarytmowaniu:

\[
\ln(T(t)-T_{env}) = \beta_0 + \beta_1 t
\]

oraz:

\[
k=-\beta_1
\]


## **17. Macierz projektu**


In [ ]:
function prepare_linearized_problem(data::MeasurementSet, ambient::Real)
    delta = data.temperature .- ambient
    valid = delta .> 0

    t = data.time[valid]
    y = log.(delta[valid])

    X = hcat(ones(length(t)), t)

    return X, y
end

X, y = prepare_linearized_problem(data, model.ambient)

@show size(X)
@show length(y)


## **18. Least squares przez `\`**


In [ ]:
β = X \ y

estimated_intercept = β[1]
estimated_slope = β[2]
estimated_k = -estimated_slope

@show β
@show estimated_k
@show model.k


### Ważne

Nie używamy `inv(X'X) * X'y`.

Używamy:

```julia
X \ y
```


## **19. Funkcja `estimate_k`**


In [ ]:
function estimate_k(data::MeasurementSet, ambient::Real)
    X, y = prepare_linearized_problem(data, ambient)
    β = X \ y
    k_est = -β[2]

    k_est > 0 ||
        throw(ArgumentError("estymowany k nie jest dodatni"))

    return (
        k = k_est,
        intercept = β[1],
        coefficients = β,
        residual = X * β - y,
    )
end


In [ ]:
fit = estimate_k(data, model.ambient)

@show fit.k
@show norm(fit.residual)


# Część IX — ocena estymowanego modelu


## **20. Model z estymowanym parametrem**


In [ ]:
estimated_model = ThermalModel(fit.k, model.ambient)

estimated_prediction = exact_series(
    estimated_model,
    data.temperature[1],
    data.time,
)

@show estimated_model


## **21. Metryki modelu**


In [ ]:
function model_metrics(predicted, observed)
    r = residuals(predicted, observed)

    return (
        mae = mean(abs.(r)),
        rmse = sqrt(mean(abs2, r)),
        max_abs_error = maximum(abs.(r)),
        mean_residual = mean(r),
    )
end

metrics = model_metrics(estimated_prediction, data.temperature)

@show metrics


## **22. Współczynnik `R²`**


In [ ]:
function r_squared(predicted, observed)
    ss_res = sum(abs2, observed .- predicted)
    ss_tot = sum(abs2, observed .- mean(observed))

    ss_tot == 0 && return NaN
    return 1 - ss_res / ss_tot
end

R2 = r_squared(estimated_prediction, data.temperature)

@show R2


# Część X — wydajność


## **23. Warm-up**


In [ ]:
simulate(model, EulerIntegrator(), 90.0, (0.0, 30.0), 0.01)
simulate(model, RK4Integrator(),   90.0, (0.0, 30.0), 0.01)

println("Warm-up zakończony.")


## **24. Prosty benchmark**


In [ ]:
function best_elapsed(f; samples=10)
    best = Inf

    for _ in 1:samples
        elapsed = @elapsed f()
        best = min(best, elapsed)
    end

    return best
end

euler_time = best_elapsed(
    () -> simulate(model, EulerIntegrator(), 90.0, (0.0, 30.0), 0.01)
)

rk4_time = best_elapsed(
    () -> simulate(model, RK4Integrator(), 90.0, (0.0, 30.0), 0.01)
)

@show euler_time
@show rk4_time


## **25. Alokacje**


In [ ]:
euler_alloc = @allocated simulate(
    model, EulerIntegrator(), 90.0, (0.0, 30.0), 0.01
)

rk4_alloc = @allocated simulate(
    model, RK4Integrator(), 90.0, (0.0, 30.0), 0.01
)

@show euler_alloc
@show rk4_alloc


## **26. Dokładność a koszt**


In [ ]:
function terminal_error(result::SimulationResult, model, T0)
    exact = exact_temperature(model, T0, result.time[end])
    return abs(result.temperature[end] - exact)
end

e = simulate(model, EulerIntegrator(), 90.0, (0.0, 30.0), 0.05)
r = simulate(model, RK4Integrator(),   90.0, (0.0, 30.0), 0.05)

@show terminal_error(e, model, 90.0)
@show terminal_error(r, model, 90.0)


### Wniosek

Algorytm oceniamy przez:

- czas,
- pamięć,
- dokładność,
- stabilność.


# Część XI — plotting


## **27. Pomiary i dopasowany model**


In [ ]:
if HAS_PLOTS
    p = Plots.scatter(
        data.time,
        data.temperature;
        label="pomiary",
        xlabel="Czas",
        ylabel="Temperatura [°C]",
        title="ThermalLab — dopasowanie modelu",
    )

    Plots.plot!(
        p,
        data.time,
        estimated_prediction;
        label="model",
        linewidth=2,
    )

    display(p)
else
    println("Plots.jl nie jest zainstalowany — pomijam wykres.")
end


## **28. Euler vs RK4 vs rozwiązanie dokładne**


In [ ]:
if HAS_PLOTS
    h = 0.5

    e = simulate(model, EulerIntegrator(), 90.0, (0.0, 30.0), h)
    r = simulate(model, RK4Integrator(),   90.0, (0.0, 30.0), h)
    exact = exact_series(model, 90.0, e.time)

    p = Plots.plot(
        e.time,
        exact;
        label="dokładne",
        linewidth=3,
        xlabel="Czas",
        ylabel="Temperatura [°C]",
        title="Porównanie integratorów",
    )

    Plots.plot!(p, e.time, e.temperature; label="Euler")
    Plots.plot!(p, r.time, r.temperature; label="RK4")

    display(p)
else
    println("Plots.jl nie jest zainstalowany — pomijam wykres.")
end


## **29. Residuals**


In [ ]:
if HAS_PLOTS
    r = residuals(estimated_prediction, data.temperature)

    p = Plots.scatter(
        data.time,
        r;
        xlabel="Czas",
        ylabel="Residual [°C]",
        title="Residuals modelu",
        label="residual",
    )

    display(p)
else
    println("Plots.jl nie jest zainstalowany — pomijam wykres.")
end


# Część XII — raportowanie


## **30. Raport tekstowy**


In [ ]:
function print_report(true_model, estimated_model, metrics, R2)
    println("="^60)
    println("THERMALLAB — RAPORT KOŃCOWY")
    println("="^60)

    @printf("k rzeczywiste:    %.6f\n", true_model.k)
    @printf("k estymowane:     %.6f\n", estimated_model.k)
    @printf("błąd k:           %.6f\n", abs(true_model.k - estimated_model.k))
    @printf("MAE:               %.6f\n", metrics.mae)
    @printf("RMSE:              %.6f\n", metrics.rmse)
    @printf("MAX |error|:       %.6f\n", metrics.max_abs_error)
    @printf("mean residual:     %.6f\n", metrics.mean_residual)
    @printf("R²:                %.6f\n", R2)

    println("="^60)
end

print_report(model, estimated_model, metrics, R2)


## **31. Eksport CSV**


In [ ]:
function export_csv(filename, time, measured, predicted)
    length(time) == length(measured) == length(predicted) ||
        throw(ArgumentError("serie muszą mieć tę samą długość"))

    open(filename, "w") do io
        println(io, "time,measured,predicted,residual")

        for i in eachindex(time, measured, predicted)
            r = predicted[i] - measured[i]

            println(
                io,
                time[i], ",",
                measured[i], ",",
                predicted[i], ",",
                r,
            )
        end
    end

    return filename
end


In [ ]:
csv_path = export_csv(
    "thermallab_results.csv",
    data.time,
    data.temperature,
    estimated_prediction,
)

println("Zapisano: ", csv_path)


# Część XIII — testy


## **32. Testy kontraktowe**


In [ ]:
function run_capstone_tests()
    test_model = ThermalModel(0.1, 20.0)

    @assert exact_temperature(test_model, 100.0, 0.0) == 100.0

    e = simulate(
        test_model,
        EulerIntegrator(),
        100.0,
        (0.0, 10.0),
        0.1,
    )

    r = simulate(
        test_model,
        RK4Integrator(),
        100.0,
        (0.0, 10.0),
        0.1,
    )

    exact = exact_series(test_model, 100.0, r.time)

    @assert length(e.time) == length(e.temperature)
    @assert length(r.time) == length(r.temperature)
    @assert all(diff(e.time) .> 0)
    @assert all(diff(r.time) .> 0)

    @assert r.temperature[end] < r.temperature[1]
    @assert r.temperature[end] > test_model.ambient

    @assert rmse(r.temperature, exact) <
            rmse(e.temperature, exact)

    @assert model_metrics(exact, exact).rmse == 0.0
    @assert isapprox(r_squared(exact, exact), 1.0)

    return true
end

@assert run_capstone_tests()

println("PASS — wszystkie testy Capstone zakończone sukcesem.")


## **33. Test walidacji błędnych danych**


In [ ]:
validation_test = try
    ThermalModel(-0.5, 20.0)
    false
catch e
    e isa ArgumentError
end

@assert validation_test

println("PASS — walidacja modelu działa.")


# Część XIV — analiza architektury


## **34. Modularność**

| Komponent | Odpowiedzialność |
|---|---|
| `ThermalModel` | parametry modelu |
| `MeasurementSet` | dane pomiarowe |
| `SimulationResult` | wynik integracji |
| `step` | pojedynczy krok algorytmu |
| `simulate` | przebieg symulacji |
| `estimate_k` | estymacja parametru |
| `model_metrics` | ocena jakości |
| `print_report` | prezentacja |
| `export_csv` | zapis danych |


## **35. Rozszerzalność przez dispatch**

Aby dodać nowy integrator:

```julia
struct HeunIntegrator <: AbstractIntegrator end
```

i nową metodę:

```julia
step(::HeunIntegrator, ...)
```

Nie trzeba modyfikować `simulate`.


# Część XV — zadania rozszerzające


## **36. Extension A — Heun**

Dodaj `HeunIntegrator` i porównaj Euler / Heun / RK4 dla tych samych kroków.


In [ ]:
# Twoja implementacja:


## **37. Extension B — estymacja `T_env`**

Rozszerz estymację tak, aby niewiadome były jednocześnie `k` i `T_env`.

To prowadzi do nieliniowego problemu optymalizacyjnego.


In [ ]:
# Zdefiniuj funkcję celu:
# objective(params) = ...


## **38. Extension C — dane z pliku**

Zastąp dane syntetyczne rzeczywistym plikiem.

Możliwe narzędzia:

- `DelimitedFiles`,
- `CSV.jl`,
- `DataFrames.jl`.


## **39. Extension D — własny pakiet**

Przenieś projekt do:

```text
ThermalLab/
├── Project.toml
├── src/
│   └── ThermalLab.jl
├── test/
│   └── runtests.jl
└── README.md
```


# Część XVI — kryteria zaliczenia


## **40. Rubryka Capstone**

| Kryterium | Punkty |
|---|---:|
| struktury danych | 10 |
| Euler | 8 |
| RK4 | 10 |
| multiple dispatch | 10 |
| symulacja | 10 |
| least squares | 12 |
| metryki | 8 |
| wydajność | 8 |
| wizualizacja | 8 |
| testy | 8 |
| dokumentacja i jakość kodu | 8 |
| **Razem** | **100** |


## **41. Minimalne wymagania zaliczeniowe**

Projekt zalicza Capstone, jeśli:

1. wykonuje się bez błędów;
2. Euler i RK4 działają;
3. RK4 daje mniejszy błąd w teście;
4. `estimate_k` zwraca dodatnie `k`;
5. least squares wykorzystuje `\`;
6. istnieją metryki jakości;
7. testy przechodzą;
8. kod jest podzielony na funkcje;
9. użyto multiple dispatch;
10. wynik został zinterpretowany.


# Część XVII — końcowy checkpoint


## **42. Pytania końcowe**

1. Dlaczego model, pomiary i wynik mają osobne typy?
2. Gdzie wykorzystujemy multiple dispatch?
3. Dlaczego `simulate` nie musi znać nazwy konkretnego algorytmu?
4. Co daje parametryzowany `ThermalModel{T}`?
5. Dlaczego dane walidujemy przy konstrukcji?
6. Jak generowane są pomiary syntetyczne?
7. Jak problem estymacji `k` staje się liniowy?
8. Dlaczego używamy `X \ y`?
9. Co mierzy RMSE?
10. Co mierzy `R²`?
11. Dlaczego residuals są ważne?
12. Dlaczego wydajność trzeba analizować razem z dokładnością?
13. Co mierzy `@allocated`?
14. Dlaczego `Plots.jl` jest opcjonalne?
15. Dlaczego notebook nie wykonuje automatycznie `Pkg.add`?
16. Jak dodać trzeci integrator?
17. Jak przenieść projekt do pakietu?
18. Jakie testy powinny znaleźć się w `runtests.jl`?


# Część XVIII — zakończenie kursu


## **43. Co opanowałeś w Lesson 0–14**

Po tym kursie potrafisz:

- programować idiomatycznie w Julia;
- projektować własne typy;
- używać funkcji i multiple dispatch;
- zarządzać pakietami i środowiskami;
- wizualizować dane;
- mierzyć wydajność;
- wykonywać obliczenia algebry liniowej;
- stosować faktoryzacje;
- implementować podstawowe metody numeryczne;
- zbudować kompletny projekt od modelu do raportu.


## **44. Najważniejsza lekcja Capstone**

Profesjonalny projekt łączy:

1. model danych,
2. algorytmy,
3. walidację,
4. testy,
5. analizę błędu,
6. wydajność,
7. reprodukowalne środowisko,
8. czytelną prezentację wyników.

ThermalLab jest demonstracją tego sposobu myślenia.


## **45. Co dalej?**

| Kierunek | Ekosystem Julia |
|---|---|
| ODE / scientific computing | DifferentialEquations.jl |
| optymalizacja | Optimization.jl, JuMP.jl |
| dane | DataFrames.jl, CSV.jl |
| statystyka | StatsBase.jl, GLM.jl |
| ML | MLJ.jl, Flux.jl |
| symbolic | Symbolics.jl |
| GPU | CUDA.jl |
| wizualizacja | Makie.jl |
| tworzenie pakietów | PkgTemplates.jl |


## **46. Final Challenge**

Zbuduj analogiczny projekt dla innej domeny:

- ładowanie kondensatora,
- model baterii,
- ruch tłumiony,
- populacja biologiczna,
- sygnał pomiarowy,
- temperatura CPU.

Wymagania:

1. minimum dwa integratory;
2. multiple dispatch;
3. estymacja parametru;
4. algebra liniowa;
5. minimum dwie metryki;
6. benchmark;
7. wykres;
8. testy;
9. raport końcowy.


## **47. Zakończenie**

Ukończyłeś **Julia od zera — Cartesian School**.

Najważniejszy rezultat kursu to umiejętność przejścia od:

> **problemu → modelu → kodu → obliczeń → walidacji → wyniku**

i zbudowania rozwiązania, które jest:

- czytelne,
- testowalne,
- rozszerzalne,
- reprodukowalne,
- numerycznie świadome.


## Źródła do dalszego rozwoju

- Julia Manual
- Julia Standard Library
- Pkg documentation
- LinearAlgebra
- Statistics
- Plots.jl
- DifferentialEquations.jl / SciML
- Julia Performance Tips
- Julia Package Development documentation


[← Lesson 13 — Numerical Computing](Lesson_13_Numerical_Computing_Julia_Cartesian_School_PL.ipynb)  
[Spis treści](../README.pl.md)